<a href="https://colab.research.google.com/github/anu5hkaa/AI-War-News-Analyser-System/blob/main/War%20news%20analzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import numpy as np

In [ ]:
api_key='e2a5d3214eeb40e297e0ab4e64e4b510'

In [ ]:

url = "https://newsapi.org/v2/everything"

war_keywords = [
    "war", "conflict", "military", "airstrike", "missile",
    "bomb", "attack", "invasion", "army", "troops",
    "defense", "clash", "border", "violence", "strike"
]
queries = [
    "war",
    "military conflict",
    "airstrike attack",
    "border clash",
    "missile strike",
    "army operation",
    "defense military",
    "troops conflict"
]

all_articles = []
news_list = []
for query in queries:
    for page in range(1, 6):   # 5 pages × 20 = 100 per query

        params = {
            "q": query,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 20,   # free API limit per request
            "page": page,
            "apiKey": api_key
        }

        response = requests.get(url, params=params)
        data = response.json()

        articles = data.get("articles", [])
        all_articles.extend(articles)

for article in all_articles:
    news = {
        "title": article.get("title", ""),
        "content": article.get("description", ""),
        "source": article.get("source", {}).get("name", ""),
        "url": article.get("url", ""),
        "date": article.get("publishedAt", "")
    }
    news_list.append(news)

In [ ]:

df = pd.DataFrame(news_list)

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df['content'][4]

In [ ]:
len((df))

In [ ]:
df = df[~df["title"].str.lower().str.contains("movie|film|trailer|review|box office")]

In [ ]:
len(df)

In [ ]:
from transformers import pipeline

# Load once (don’t put inside loop)
classifier = pipeline("zero-shot-classification")

def is_war_news(text):
    labels = ["war or military news", "not related"]

    result = classifier(text, labels)

    return result["labels"][0] == "war or military news"

In [ ]:
df = df[df["title"].apply(is_war_news)]

In [ ]:
print(len(df))


In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # This is a good, efficient model

df["title"] = df["title"].fillna('').str.lower()
df["content"] = df["content"].fillna('').str.lower()
df["source"] = df["source"].astype(str).str.lower()
# Prepare your text data: combine title and content or just use one
df['combined_text'] = df['title'] + " " + df['content']

# Convert the combined text into embeddings
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

In [ ]:
embeddings.shape


In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings)

# IMPORTANT → normalize for cosine similarity
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # cosine similarity
index.add(embeddings)

In [ ]:
query = "pakistan and india at war"

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding)

faiss.normalize_L2(query_embedding)

In [ ]:
D, I = index.search(query_embedding, k=5)

In [ ]:
for i in I[0]:
    print(df.iloc[i]["title"])
    print(df.iloc[i]["source"])
    print(df.iloc[i]['url'])
    print("-----")

In [ ]:
!pip install transformers



In [ ]:
!pip install feedparser

In [ ]:
import numpy as np
import pandas as pd
import faiss
import feedparser
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder


model = SentenceTransformer('all-MiniLM-L6-v2')
cross_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
df['content'] = df['content'].fillna('').astype(str).str.lower()
df['title'] = df['title'].fillna('').astype(str).str.lower()
df['date'] = df['date'].astype(str)

df['full_text'] = df['title'] + " " + df['content']

embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)


war_keywords = [
    "war", "conflict", "attack", "military", "missile",
    "battle", "army", "troops", "airstrike", "bomb",
    "border", "clash", "violence", "defense", "ceasefire"
]
def expand_query(query):
    return query + " war conflict military attack"


def smart_filter(results, query):
    words = [w for w in query.split() if len(w) > 2]

    filtered = []

    for item in results:
        text = (item["title"] + " " + item["content"]).lower()

        # must be war-related
        if not any(k in text for k in war_keywords):
            continue

        if len(words) == 1:
            # single word → flexible
            if words[0] in text:
                filtered.append(item)
        else:
            # multiple words → strict (ALL must match)
            if all(word in text for word in words):
                filtered.append(item)

    return filtered


def clean_google_link(link):
    try:
        response = requests.get(link, allow_redirects=True, timeout=5)
        return response.url
    except:
        return link

def fetch_rss_news(query):

    query_formatted = query.replace(" ", "+")

    rss_urls = [
        f"https://news.google.com/rss/search?q={query_formatted}+war",
        f"https://news.google.com/rss/search?q={query_formatted}+conflict",
        f"https://news.google.com/rss/search?q={query_formatted}+military",

        "https://www.defensenews.com/arc/outboundfeeds/rss/?outputType=xml",
        "https://www.militarytimes.com/arc/outboundfeeds/rss/",
        "https://www.armytimes.com/arc/outboundfeeds/rss/"
    ]

    rss_data = []

    for url in rss_urls:
        feed = feedparser.parse(url)

        for entry in feed.entries:
            text = entry.title.lower()

            if any(k in text for k in war_keywords):
                rss_data.append({
                    "title": entry.title.lower(),
                    "content": entry.title,
                    "url": clean_google_link(entry.link),
                    "date": entry.published if "published" in entry else "",
                    "full_text": entry.title.lower()
                })

    seen = set()
    unique = []

    for item in rss_data:
        if item["url"] not in seen:
            unique.append(item)
            seen.add(item["url"])

    return unique[:40]

def get_top_matches(query):

    original_query = query.lower().strip()
    expanded_query = expand_query(original_query)

    print("Searching...")


    query_embedding = model.encode([expanded_query])
    query_embedding = np.array(query_embedding)
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k=20)

    api_results = []

    for i in I[0]:
        if i < len(df):
            api_results.append({
                "title": str(df.iloc[i]['title']),
                "content": str(df.iloc[i]['content'])[:1000],
                "url": df.iloc[i]['url'],
                "date": df.iloc[i]['date']
            })


    rss_results = fetch_rss_news(original_query)

    api_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in api_results]
    api_scores = cross_model.predict(api_pairs) if api_pairs else []
    api_ranked = sorted(zip(api_scores, api_results), reverse=True)

    rss_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in rss_results]
    rss_scores = cross_model.predict(rss_pairs) if rss_pairs else []
    rss_ranked = sorted(zip(rss_scores, rss_results), reverse=True)

    combined = [item for _, item in (api_ranked[:5] + rss_ranked[:5])]

    filtered = smart_filter(combined, original_query)

    if not filtered:
        print("Not Found")
        return []

    final_results = filtered[:10]

    clean_results = []

    for item in final_results:
        try:
            parsed = pd.to_datetime(item["date"], errors='coerce')
            date_str = parsed.strftime("%Y-%m-%d") if not pd.isna(parsed) else str(item["date"])
        except:
            date_str = str(item["date"])

        clean_results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"],
            "date": date_str
        })

    if len(clean_results) >= 2:
        prompt = create_prompt(original_query, clean_results)
        summary = get_llm_response(prompt)

        print("\nSummary:\n")
        print(summary)

    print("\nSources:\n")

    for i, news in enumerate(clean_results):
        print(f"{i+1}. {news['title']}")
        print(news["url"])
        print("Date:", news["date"])
        print()


In [ ]:
!pip install Groq

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"]=getpass("Enter your Groq API key: ")

In [ ]:
from groq import Groq
client=Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
def create_prompt(query, clean_results):

    prompt = f"""
You are a war news analyst.

User Query: {query}

Below are some news articles:

"""

    for i, news in enumerate(clean_results, 1):
        prompt += f"""
Article {i}:
Title: {news['title']}
Content: {news['content']}
Date: {news['date']}
"""

    prompt += """
Task:
- Summarize what is happening
- Explain in simple simple words
-Key insights:
- Keep answer short
"""

    return prompt

In [ ]:
def get_llm_response(prompt):

    print("LLM running...")

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    print("LLM done!")

    return response.choices[0].message.content

In [ ]:
get_top_matches("iran and us")